In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMClassifier

In [2]:
train_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e9/test.csv")

In [3]:
train_df.head()

,id,Age,Annual_Income_USD,Daily_Commute_km,Number_of_Cars_Owned,Charging_Stations_Near_Home,Charging_Stations_Near_Work,Environmental_Concern_Level,Gender,City_Type,Current_Car_Type,Home_Charging_Possible,Subsidy_Available,Range_Anxiety_Level,Will_Buy_EV
0,0,66,92887.0,23.4,2,3,7,1.0,Male,Suburban,Sedan,Yes,No,Low,No
1,1,38,30000.0,5.0,1,2,2,4.0,Male,Rural,SUV,Yes,No,Low,No
2,2,26,94389.0,36.8,1,8,15,5.0,Female,Urban,Sedan,No,Yes,Low,Yes
3,3,66,73580.0,23.7,2,6,9,3.0,Male,Suburban,Hatchback,Yes,No,Low,No
4,4,54,57898.0,50.8,1,2,3,3.0,Male,Suburban,Hatchback,Yes,No,Low,No


In [4]:
X = train_df.drop(columns=['id','Will_Buy_EV'])
Y = train_df['Will_Buy_EV']

In [5]:
print(Y.value_counts())
print(Y.value_counts(normalize=True))

Will_Buy_EV
No     551886
Yes    116779
Name: count, dtype: int64
Will_Buy_EV
No     0.825355
Yes    0.174645
Name: proportion, dtype: float64


In [6]:
scale_pos_weight = 551886 / 116779

In [7]:
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

In [8]:
print(cat_cols)
print(num_cols)

Index(['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level'],
      dtype='object')
Index(['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
       'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work',
       'Environmental_Concern_Level'],
      dtype='object')


In [9]:
cat_pipeline = Pipeline([
    ('LabelEncoder',OrdinalEncoder()),
])

preprocessor = ColumnTransformer([
    ('categoric',cat_pipeline,cat_cols)
],remainder='passthrough'
)

In [10]:
model = Pipeline([
    ('preprocessor', preprocessor),

    ('lgbm', LGBMClassifier(
        objective='binary',
        n_estimators=2000,
        learning_rate=0.02,

        num_leaves=20,
        max_depth=6,
        min_child_samples=40,

        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,

        reg_alpha=0.2,
        reg_lambda=2.0,
        min_split_gain=0.01,

        scale_pos_weight=scale_pos_weight,

        random_state=42,
        n_jobs=-1,
        verbosity=-1
    ))
])

In [11]:
model.fit(X,Y)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categoric',
                                                  Pipeline(steps=[('LabelEncoder',
                                                                   OrdinalEncoder())]),
                                                  Index(['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
       'Subsidy_Available', 'Range_Anxiety_Level'],
      dtype='object'))])),
                ('lgbm',
                 LGBMClassifier(colsample_bytree=0.8, learning_rate=0.02,
                                max_depth=6, min_child_samples=40,
                                min_split_gain=0.01, n_estimators=2000,
                                n_jobs=-1, num_leaves=20, objective='binary',
                                random_state=42, reg_alpha=0.2, reg_lambda=2.0,
                                scale_pos_weight=4.72590106097843,
                                subsample=0.8, subsample_freq=1,
                                verbosity=-1))])

In [12]:
y_preds = model.predict_proba(test_df.drop(columns=['id']))[:,1]

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [13]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'Will_Buy_EV':y_preds
})

submission.to_csv('submission.csv', index=False)